In [3]:
from pathlib import Path
import math

# ====== Input paths ======
AUDIO_DIR = Path("/home/danila/networks/data/audio")   # mp3
DATA_DIR  = Path("/home/danila/networks/data")

TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

# ====== Output paths ======
OUT_DIR   = DATA_DIR / "embeddings" / "audio_wavlm_large_fps5_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / "index.parquet"

# ====== Audio / alignment ======
TARGET_SR = 16_000

TARGET_FPS = 5
HOP_SEC = 1.0 / TARGET_FPS
# bins: [t, t+HOP_SEC) с timestamp = t

# ====== Chunking for long audio (important) ======
CHUNK_SEC   = 20.0
OVERLAP_SEC = 2.0

# ====== Model ======
MODEL_NAME = "microsoft/wavlm-large"

# ====== Runtime ======
DEVICE = "cuda"
USE_AMP = True    # mixed precision on GPU

# ====== Saving ======
EMB_SAVE_DTYPE = "float16"
SKIP_IF_EXISTS = True
ID_WIDTH = 5


In [4]:
import json
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def _ts():
    return datetime.now().strftime("%H:%M:%S")

In [5]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

all_ids = sorted(pd.concat([train_df["Filename"], valid_df["Filename"]]).unique().tolist())
tqdm.write(f"[{_ts()}] Total ids from splits: {len(all_ids)}")
tqdm.write(f"Examples: {all_ids[:10]}")


[18:18:09] Total ids from splits: 12660
Examples: ['00000', '00001', '00002', '00003', '00004', '00005', '00006', '00007', '00008', '00009']


In [6]:
def load_audio_mono_16k(mp3_path: Path, target_sr: int = 16000):
    """
    Returns:
      wav: torch.FloatTensor [T] in range ~[-1, 1]
      sr: int
    """
    # torchaudio
    try:
        import torchaudio
        wav, sr = torchaudio.load(str(mp3_path))  # [C, T]
        if wav.ndim == 2 and wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)  # [T]

        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
            wav = resampler(wav.unsqueeze(0)).squeeze(0)
            sr = target_sr

        return wav.contiguous().float(), sr

    except Exception as e_ta:
        # fallback: librosa
        try:
            import librosa
            x, sr = librosa.load(str(mp3_path), sr=target_sr, mono=True)
            wav = torch.from_numpy(x).float()
            return wav.contiguous(), target_sr
        except Exception as e_lr:
            raise RuntimeError(f"Failed to load mp3 with torchaudio ({e_ta}) and librosa ({e_lr})")

In [7]:
from transformers import AutoFeatureExtractor, WavLMModel

assert torch.cuda.is_available() or DEVICE == "cpu", "CUDA not available; set DEVICE='cpu' or install CUDA torch."

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = WavLMModel.from_pretrained(MODEL_NAME)
model.eval()
model.to(DEVICE)

HIDDEN = int(model.config.hidden_size)
tqdm.write(f"[{_ts()}] Loaded {MODEL_NAME} | hidden={HIDDEN} | device={DEVICE}")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

[18:18:47] Loaded microsoft/wavlm-large | hidden=1024 | device=cuda


In [8]:
def make_time_bins(duration_sec: float, hop_sec: float):
    """
    bins: [t, t+hop) , timestamps = t
    num_bins = ceil(duration / hop)
    """
    num_bins = int(math.ceil(duration_sec / hop_sec))
    timestamps = (np.arange(num_bins, dtype=np.float32) * hop_sec)
    return num_bins, timestamps

@torch.inference_mode()
def wavlm_forward_hidden(wav_1d: torch.Tensor, sr: int):
    """
    wav_1d: [T] float32 on CPU
    returns hidden: [L, H] on GPU (dtype может быть fp16 under AMP)
    and stride_sec: seconds per hidden frame (оценка)
    """
    # feature_extractor expects np arrays or lists
    inputs = feature_extractor(
        wav_1d.numpy(),
        sampling_rate=sr,
        return_tensors="pt",
        padding=False,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    if DEVICE == "cuda" and USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(**inputs)
    else:
        out = model(**inputs)

    hidden = out.last_hidden_state.squeeze(0)  # [L, H]
    # how many seconds per 1 hidden-frame
    L = hidden.shape[0]
    stride_sec = (float(wav_1d.numel()) / float(sr)) / max(L, 1)
    return hidden, stride_sec

In [9]:
def process_one_audio(video_id: str):
    mp3_path = AUDIO_DIR / f"{video_id}.mp3"
    if not mp3_path.exists():
        return None, {"video_id": video_id, "reason": f"mp3_not_found: {mp3_path}"}

    out_npz  = OUT_DIR / f"{video_id}.npz"
    out_meta = OUT_DIR / f"{video_id}.json"

    if SKIP_IF_EXISTS and out_npz.exists() and out_meta.exists():
        try:
            d = json.loads(out_meta.read_text())
            d["video_id"] = d.get("video_id", video_id)
            d["npz_path"] = str(out_npz)
            return d, None
        except Exception:
            return {"video_id": video_id, "npz_path": str(out_npz), "skipped": True}, None

    # load audio
    wav, sr = load_audio_mono_16k(mp3_path, TARGET_SR)
    duration_sec = float(wav.numel()) / float(sr)

    num_bins, timestamps = make_time_bins(duration_sec, HOP_SEC)

    # accumulators on GPU (fp32)
    sums = torch.zeros((num_bins, HIDDEN), device=DEVICE, dtype=torch.float32)
    cnts = torch.zeros((num_bins,), device=DEVICE, dtype=torch.int32)

    chunk_samples   = int(CHUNK_SEC * sr)
    overlap_samples = int(OVERLAP_SEC * sr)
    step_samples    = max(1, chunk_samples - overlap_samples)

    wav_len = wav.numel()
    chunk_starts = list(range(0, wav_len, step_samples))

    for cs in chunk_starts:
        ce = min(wav_len, cs + chunk_samples)
        wav_chunk = wav[cs:ce]

        # forward
        hidden, stride_sec = wavlm_forward_hidden(wav_chunk, sr)  # hidden [L,H] on GPU
        L = hidden.shape[0]

        # the time of each hidden-frame in the global scale
        # t = chunk_start_sec + i * stride_sec
        chunk_start_sec = float(cs) / float(sr)
        frame_times = chunk_start_sec + torch.arange(L, device=DEVICE, dtype=torch.float32) * float(stride_sec)

        # convert to bin 0.2 sec.
        bin_idx = torch.floor(frame_times / float(HOP_SEC)).to(torch.long)

        valid = (bin_idx >= 0) & (bin_idx < num_bins)
        if valid.any():
            idx = bin_idx[valid]
            h   = hidden[valid].float()  # -> fp32

            sums.index_add_(0, idx, h)
            cnts.index_add_(0, idx, torch.ones_like(idx, dtype=torch.int32))

    # finalize
    cnts_cpu = cnts.detach().cpu().numpy()
    valid_bins = cnts_cpu > 0

    emb = sums / (cnts.clamp_min(1).unsqueeze(1).float())
    emb = emb.detach().cpu().numpy()

    # cast for saving
    if EMB_SAVE_DTYPE == "float16":
        emb_save = emb.astype(np.float16)
    else:
        emb_save = emb.astype(np.float32)

    # timestamps for bins
    bin_start_sec = timestamps
    bin_end_sec   = (timestamps + HOP_SEC).astype(np.float32)

    meta = {
        "video_id": video_id,
        "mp3_path": str(mp3_path),
        "npz_path": str(out_npz),
        "model": MODEL_NAME,
        "sr": int(sr),
        "duration_sec": float(duration_sec),
        "hop_sec": float(HOP_SEC),
        "num_bins": int(num_bins),
        "hidden_size": int(HIDDEN),
        "chunk_sec": float(CHUNK_SEC),
        "overlap_sec": float(OVERLAP_SEC),
        "valid_bins": int(valid_bins.sum()),
        "save_dtype": EMB_SAVE_DTYPE,
    }

    np.savez_compressed(
        out_npz,
        timestamps_sec=bin_start_sec,
        bin_start_sec=bin_start_sec,
        bin_end_sec=bin_end_sec,
        valid=valid_bins.astype(np.bool_),
        counts=cnts_cpu.astype(np.int32),
        embeddings=emb_save,
    )
    out_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

    return meta, None

In [10]:
metas = []
failed = []

pbar = tqdm(all_ids, desc="Extract audio embeddings (WavLM-Large)")
total = len(all_ids)

for i, vid in enumerate(pbar, start=1):
    try:
        meta, err = process_one_audio(vid)
        if err is not None:
            failed.append(err)
            tqdm.write(f"[{_ts()}] ERROR {vid}: {err['reason']}")
        else:
            metas.append(meta)
    except Exception as e:
        failed.append({"video_id": vid, "reason": repr(e)})
        tqdm.write(f"[{_ts()}] ERROR {vid}: {repr(e)}")
        tqdm.write(traceback.format_exc().rstrip())

    if i % 50 == 0:
        tqdm.write(f"[{_ts()}] progress {i}/{total} ok={len(metas)} failed={len(failed)}")

Extract audio embeddings (WavLM-Large):   0%|          | 0/12660 [00:00<?, ?it/s]

/home/danila/networks/.venv/lib/python3.12/site-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


[18:20:40] progress 50/12660 ok=50 failed=0
[18:20:44] progress 100/12660 ok=100 failed=0
[18:20:47] progress 150/12660 ok=150 failed=0
[18:20:50] progress 200/12660 ok=200 failed=0
[18:20:53] progress 250/12660 ok=250 failed=0
[18:20:56] progress 300/12660 ok=300 failed=0
[18:20:59] progress 350/12660 ok=350 failed=0
[18:21:02] progress 400/12660 ok=400 failed=0
[18:21:05] progress 450/12660 ok=450 failed=0
[18:21:08] progress 500/12660 ok=500 failed=0
[18:21:11] progress 550/12660 ok=550 failed=0
[18:21:15] progress 600/12660 ok=600 failed=0
[18:21:17] progress 650/12660 ok=650 failed=0
[18:21:21] progress 700/12660 ok=700 failed=0
[18:21:24] progress 750/12660 ok=750 failed=0
[18:21:26] progress 800/12660 ok=800 failed=0
[18:21:29] progress 850/12660 ok=850 failed=0
[18:21:32] progress 900/12660 ok=900 failed=0
[18:21:35] progress 950/12660 ok=950 failed=0
[18:21:37] progress 1000/12660 ok=1000 failed=0
[18:21:40] progress 1050/12660 ok=1050 failed=0
[18:21:43] progress 1100/12660 o

In [11]:
import json
from pathlib import Path

meta_rows = []
json_files = sorted(Path(OUT_DIR).glob("*.json"))

for jf in tqdm(json_files, desc="Load saved audio metas"):
    try:
        d = json.loads(jf.read_text())
        vid = d.get("video_id", jf.stem)
        d["video_id"] = vid
        npz_path = Path(d.get("npz_path", OUT_DIR / f"{vid}.npz"))
        d["npz_path"] = str(npz_path)
        d["npz_exists"] = npz_path.exists()
        meta_rows.append(d)
    except Exception as e:
        tqdm.write(f"[{_ts()}] meta read error {jf.name}: {repr(e)}")

meta_df = pd.DataFrame(meta_rows)

fail_path = Path(OUT_DIR) / "failed.csv"
if fail_path.exists():
    fail_df = pd.read_csv(fail_path, dtype={"video_id": str})
else:
    fail_df = pd.DataFrame(columns=["video_id", "reason"])

meta_df.to_parquet(INDEX_PATH, index=False)
tqdm.write(f"[{_ts()}] Saved index: {INDEX_PATH}")
tqdm.write(f"[{_ts()}] DONE | ok={len(meta_df)} failed={len(fail_df)} metas_json={len(json_files)}")

Load saved audio metas:   0%|          | 0/12658 [00:00<?, ?it/s]

[18:33:35] Saved index: /home/danila/networks/data/embeddings/audio_wavlm_large_fps5_v1/index.parquet
[18:33:35] DONE | ok=12658 failed=0 metas_json=12658


## Emotional embeddings

In [12]:
from pathlib import Path

MODEL_NAME_ER = "superb/wav2vec2-large-superb-er"

OUT_DIR_ER = DATA_DIR / "embeddings" / "audio_superb_wav2vec2_large_er_fps5_v1"
OUT_DIR_ER.mkdir(parents=True, exist_ok=True)

INDEX_PATH_ER = OUT_DIR_ER / "index.parquet"

EMB_SAVE_DTYPE_ER = "float16"
SKIP_IF_EXISTS_ER = True

In [13]:
import torch
from transformers import AutoFeatureExtractor, AutoModel, AutoModelForAudioClassification

feature_extractor_er = AutoFeatureExtractor.from_pretrained(MODEL_NAME_ER)

try:
    model_er = AutoModel.from_pretrained(MODEL_NAME_ER)
except Exception:
    model_er = AutoModelForAudioClassification.from_pretrained(MODEL_NAME_ER)

model_er.eval().to(DEVICE)

base_er = getattr(model_er, "wav2vec2", None)
if base_er is None:
    base_er = getattr(model_er, "model", None)

if base_er is None:
    base_er = model_er

# Finding out the hidden size
HIDDEN_ER = int(getattr(base_er.config, "hidden_size", getattr(model_er.config, "hidden_size", None)))
print(f"[{_ts()}] Loaded {MODEL_NAME_ER} | hidden={HIDDEN_ER} | device={DEVICE}")


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

[18:34:18] Loaded superb/wav2vec2-large-superb-er | hidden=1024 | device=cuda


In [14]:
@torch.inference_mode()
def superb_er_forward_hidden(wav_1d: torch.Tensor, sr: int):
    """
    wav_1d: [T] float32 on CPU
    returns hidden: [L, H] on GPU and stride_sec approx
    """
    inputs = feature_extractor_er(
        wav_1d.numpy(),
        sampling_rate=sr,
        return_tensors="pt",
        padding=False,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    if DEVICE == "cuda" and USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = base_er(**inputs, return_dict=True)
    else:
        out = base_er(**inputs, return_dict=True)

    # In most cases, there is a last_hidden_state
    hidden = getattr(out, "last_hidden_state", None)
    if hidden is None:
        hs = getattr(out, "hidden_states", None)
        if hs is not None and len(hs) > 0:
            hidden = hs[-1]
        else:
            raise RuntimeError("Could not obtain hidden states from SUPERB-ER model output")

    hidden = hidden.squeeze(0)  # [L, H]

    L = hidden.shape[0]
    stride_sec = (float(wav_1d.numel()) / float(sr)) / max(L, 1)
    return hidden, stride_sec

In [15]:
import numpy as np
import json

def process_one_audio_er(video_id: str):
    mp3_path = AUDIO_DIR / f"{video_id}.mp3"
    if not mp3_path.exists():
        return None, {"video_id": video_id, "reason": f"mp3_not_found: {mp3_path}"}

    out_npz  = OUT_DIR_ER / f"{video_id}.npz"
    out_meta = OUT_DIR_ER / f"{video_id}.json"

    if SKIP_IF_EXISTS_ER and out_npz.exists() and out_meta.exists():
        try:
            d = json.loads(out_meta.read_text())
            d["video_id"] = d.get("video_id", video_id)
            d["npz_path"] = str(out_npz)
            return d, None
        except Exception:
            return {"video_id": video_id, "npz_path": str(out_npz), "skipped": True}, None

    wav, sr = load_audio_mono_16k(mp3_path, TARGET_SR)
    duration_sec = float(wav.numel()) / float(sr)

    num_bins, timestamps = make_time_bins(duration_sec, HOP_SEC)

    sums = torch.zeros((num_bins, HIDDEN_ER), device=DEVICE, dtype=torch.float32)
    cnts = torch.zeros((num_bins,), device=DEVICE, dtype=torch.int32)

    chunk_samples   = int(CHUNK_SEC * sr)
    overlap_samples = int(OVERLAP_SEC * sr)
    step_samples    = max(1, chunk_samples - overlap_samples)

    wav_len = wav.numel()
    chunk_starts = list(range(0, wav_len, step_samples))

    for cs in chunk_starts:
        ce = min(wav_len, cs + chunk_samples)
        wav_chunk = wav[cs:ce]

        hidden, stride_sec = superb_er_forward_hidden(wav_chunk, sr)  # [L,H] on GPU
        L = hidden.shape[0]

        chunk_start_sec = float(cs) / float(sr)
        frame_times = chunk_start_sec + torch.arange(L, device=DEVICE, dtype=torch.float32) * float(stride_sec)

        bin_idx = torch.floor(frame_times / float(HOP_SEC)).to(torch.long)

        valid = (bin_idx >= 0) & (bin_idx < num_bins)
        if valid.any():
            idx = bin_idx[valid]
            h   = hidden[valid].float()

            sums.index_add_(0, idx, h)
            cnts.index_add_(0, idx, torch.ones_like(idx, dtype=torch.int32))

    cnts_cpu = cnts.detach().cpu().numpy()
    valid_bins = cnts_cpu > 0

    emb = (sums / (cnts.clamp_min(1).unsqueeze(1).float())).detach().cpu().numpy()

    if EMB_SAVE_DTYPE_ER == "float16":
        emb_save = emb.astype(np.float16)
    else:
        emb_save = emb.astype(np.float32)

    bin_start_sec = timestamps.astype(np.float32)
    bin_end_sec   = (timestamps + HOP_SEC).astype(np.float32)

    meta = {
        "video_id": video_id,
        "mp3_path": str(mp3_path),
        "npz_path": str(out_npz),
        "model": MODEL_NAME_ER,
        "sr": int(sr),
        "duration_sec": float(duration_sec),
        "hop_sec": float(HOP_SEC),
        "num_bins": int(num_bins),
        "hidden_size": int(HIDDEN_ER),
        "chunk_sec": float(CHUNK_SEC),
        "overlap_sec": float(OVERLAP_SEC),
        "valid_bins": int(valid_bins.sum()),
        "save_dtype": EMB_SAVE_DTYPE_ER,
    }

    np.savez_compressed(
        out_npz,
        timestamps_sec=bin_start_sec,
        bin_start_sec=bin_start_sec,
        bin_end_sec=bin_end_sec,
        valid=valid_bins.astype(np.bool_),
        counts=cnts_cpu.astype(np.int32),
        embeddings=emb_save,
    )
    out_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

    return meta, None

In [16]:
import traceback
from tqdm.auto import tqdm

metas_er = []
failed_er = []

pbar = tqdm(all_ids, desc="Extract audio embeddings (SUPERB ER)")
total = len(all_ids)

for i, vid in enumerate(pbar, start=1):
    try:
        meta, err = process_one_audio_er(vid)
        if err is not None:
            failed_er.append(err)
            tqdm.write(f"[{_ts()}] ERROR ER {vid}: {err['reason']}")
        else:
            metas_er.append(meta)
    except Exception as e:
        failed_er.append({"video_id": vid, "reason": repr(e)})
        tqdm.write(f"[{_ts()}] ERROR ER {vid}: {repr(e)}")
        tqdm.write(traceback.format_exc().rstrip())

    if i % 50 == 0:
        tqdm.write(f"[{_ts()}] progress ER {i}/{total} ok={len(metas_er)} failed={len(failed_er)}")

# сохраним failed.csv на всякий
if len(failed_er):
    pd.DataFrame(failed_er).to_csv(OUT_DIR_ER / "failed.csv", index=False)
    tqdm.write(f"[{_ts()}] Saved failures: {OUT_DIR_ER / 'failed.csv'}")

Extract audio embeddings (SUPERB ER):   0%|          | 0/12660 [00:00<?, ?it/s]

[18:37:09] progress ER 50/12660 ok=50 failed=0
[18:37:11] progress ER 100/12660 ok=100 failed=0
[18:37:13] progress ER 150/12660 ok=150 failed=0
[18:37:15] progress ER 200/12660 ok=200 failed=0
[18:37:18] progress ER 250/12660 ok=250 failed=0
[18:37:20] progress ER 300/12660 ok=300 failed=0
[18:37:22] progress ER 350/12660 ok=350 failed=0
[18:37:24] progress ER 400/12660 ok=400 failed=0
[18:37:27] progress ER 450/12660 ok=450 failed=0
[18:37:29] progress ER 500/12660 ok=500 failed=0
[18:37:31] progress ER 550/12660 ok=550 failed=0
[18:37:34] progress ER 600/12660 ok=600 failed=0
[18:37:36] progress ER 650/12660 ok=650 failed=0
[18:37:38] progress ER 700/12660 ok=700 failed=0
[18:37:41] progress ER 750/12660 ok=750 failed=0
[18:37:43] progress ER 800/12660 ok=800 failed=0
[18:37:45] progress ER 850/12660 ok=850 failed=0
[18:37:47] progress ER 900/12660 ok=900 failed=0
[18:37:49] progress ER 950/12660 ok=950 failed=0
[18:37:51] progress ER 1000/12660 ok=1000 failed=0
[18:37:54] progress 

In [17]:
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

meta_rows = []
json_files = sorted(Path(OUT_DIR_ER).glob("*.json"))

for jf in tqdm(json_files, desc="Load saved ER metas"):
    try:
        d = json.loads(jf.read_text())
        vid = d.get("video_id", jf.stem)
        d["video_id"] = vid

        npz_path = Path(d.get("npz_path", OUT_DIR_ER / f"{vid}.npz"))
        d["npz_path"] = str(npz_path)
        d["npz_exists"] = npz_path.exists()

        meta_rows.append(d)
    except Exception as e:
        tqdm.write(f"[{_ts()}] meta read error ER {jf.name}: {repr(e)}")

meta_df_er = pd.DataFrame(meta_rows)
meta_df_er.to_parquet(INDEX_PATH_ER, index=False)

tqdm.write(f"[{_ts()}] Saved ER index: {INDEX_PATH_ER}")
tqdm.write(f"[{_ts()}] DONE ER | ok={len(meta_df_er)} metas_json={len(json_files)}")

Load saved ER metas:   0%|          | 0/12658 [00:00<?, ?it/s]

[18:46:49] Saved ER index: /home/danila/networks/data/embeddings/audio_superb_wav2vec2_large_er_fps5_v1/index.parquet
[18:46:49] DONE ER | ok=12658 metas_json=12658
